In [ ]:
%pip install -q -U torch torchvision "transformers>=4.56.0,<5" accelerate "mineru-vl-utils[transformers]" openai pymupdf pillow pandas

In [1]:
import ast, json, re
from itertools import chain, count
from pathlib import Path
import pandas as pd, pymupdf, torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from mineru_vl_utils import MinerUClient
from openai import OpenAI
from PIL import Image

In [ ]:
vlm = Qwen2VLForConditionalGeneration.from_pretrained("opendatalab/MinerU2.5-2509-1.2B", dtype=torch.bfloat16, device_map="auto")
processor = AutoProcessor.from_pretrained("opendatalab/MinerU2.5-2509-1.2B", use_fast=True)
mineru = MinerUClient(backend="transformers", model=vlm, processor=processor)

In [2]:
client = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="YOUR_API_KEY_HERE")

In [3]:
PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"
PDFS_DIR = DATA_DIR / "raw" / "provas"
MINERU_DIR = DATA_DIR / "interim" / "bboxes" / "mineru"
GEMINI_DIR = DATA_DIR / "interim" / "bboxes" / "gemini"
for d in (MINERU_DIR, GEMINI_DIR):
    d.mkdir(parents=True, exist_ok=True)
PDFS = sorted(p for p in PDFS_DIR.iterdir() if p.suffix.lower() == ".pdf")
NORM, Y_TOL = 1000, 20
IMAGE_CATEGORIES = {"image", "figure"}

In [4]:
SYSTEM = """You are a precise document-layout analysis system. Identify semantic blocks in an exam from a CSV with columns [id, page, category, text, bbox].
Output formats:
- ["continuation", start_id, end_id]
- ["context", start_id, end_id, [question_start_id, ...]]
- ["question", start_id, end_id]

Block types:
- "continuation": content at the start of a page belonging to a block started on a previous page.
- "context": shared stimulus/reference material used by one or more questions. It is not itself a question.
- "question": an actual question, starting at its question number/title and including all associated content and alternatives.

Rules:
1. Ignore headers, footers, page numbers, watermarks, and decorative elements.
2. A question starts at its question number/title.
3. If the first content on a page continues a previous block, classify it as "continuation".
4. Shared stimulus/reference material for one or more questions is "context".
5. A context can serve multiple questions (list their start IDs as the 4th element).
6. Do not classify material as context merely because it precedes a question.
7. Group all elements of the same semantic block into one range.
8. Every block must have a distinct range [start_id, end_id].
9. Do not merge unrelated blocks.
10. Preserve original id order.
11. Output only meaningful blocks; ignore noise or page metadata.
12. Output plain text only: no prose, no code fences, no backticks.

Example:
[["context", 10, 20, [21, 26]], ["question", 21, 25], ["question", 26, 30], ["continuation", 31, 35], ["question", 36, 40]]"""

In [5]:
def flat(pages):
    return list(chain.from_iterable(pages))


def render(page):
    pix = page.get_pixmap(alpha=False)
    return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)


def order_page(blocks):
    rows = []
    for b in sorted(blocks, key=lambda b: b["bbox"][1]):
        row = next((r for r in rows if abs(b["bbox"][1] - r[0]["bbox"][1]) <= Y_TOL), None)
        if row is None:
            rows.append([b])
        else:
            row.append(b)
    return [b for r in rows for b in sorted(r, key=lambda b: b["bbox"][0])]


def dump_pages(pages):
    j = lambda o: json.dumps(o, ensure_ascii=False, separators=(",", ":"))
    return "[\n" + ",\n".join("[\n" + ",\n".join(j(b) for b in p) + "\n]" for p in pages) + "\n]\n"


def dump_blocks(blocks):
    return "[\n" + ",\n".join(json.dumps(b, ensure_ascii=False, separators=(",", ":")) for b in blocks) + "\n]\n"


def extract_layout(pdf, batch=8):
    path = MINERU_DIR / f"{pdf.stem}.json"
    if path.exists():
        pages = json.loads(path.read_text("utf-8"))
    else:
        ids, pages = count(), []
        with pymupdf.open(pdf) as doc:
            for start in range(0, len(doc), batch):
                imgs = [render(doc[i]) for i in range(start, min(start + batch, len(doc)))]
                for i, blocks in enumerate(mineru.batch_two_step_extract(imgs), start):
                    pages.append([{"id": next(ids), "page": i, "category": b.type, "text": b.content or "", "bbox": [round(v * NORM) for v in b.bbox]} for b in blocks])
        path.write_text(dump_pages(pages), "utf-8")
    pages = [order_page(p) for p in pages]
    for i, b in enumerate(flat(pages)):
        b["id"] = i
    return pages


def segment_blocks(pages):
    prompt = pd.DataFrame(flat(pages)).to_csv(index=False)
    response = client.chat.completions.create(model="gemini-3.6-flash", messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}], temperature=0.0)
    return ast.literal_eval(re.sub(r"```(?:\w+)?\s*|\s*```", "", response.choices[0].message.content).strip())


def block_pieces(elements, start, end):
    els = elements[start:end + 1]
    pieces = []
    for page in sorted({e["page"] for e in els}):
        on_page = [e for e in els if e["page"] == page]
        x0, y0, x1, y1 = zip(*(e["bbox"] for e in on_page))
        pieces.append({"page": page, "bbox": [min(x0), min(y0), max(x1), max(y1)], "images": [e["bbox"] for e in on_page if e["category"] in IMAGE_CATEGORIES]})
    return pieces


def build_blocks(elements, segments):
    blocks, number = [], 0
    for kind, start, end, *extra in segments:
        block = {"type": kind}
        if kind == "question":
            number += 1
            block["number"] = number
        block.update(start=start, end=end)
        if kind == "context":
            block["questions"] = extra[0]
        block["pieces"] = block_pieces(elements, start, end)
        blocks.append(block)
    return blocks

In [6]:
for pdf in PDFS:
    path = GEMINI_DIR / f"{pdf.stem}.json"
    if path.exists():
        continue
    pages = extract_layout(pdf)
    blocks = build_blocks(flat(pages), segment_blocks(pages))
    path.write_text(dump_blocks(blocks), "utf-8")
    print(pdf.stem, sum(b["type"] == "question" for b in blocks))